# LAB 08 — BÀI TẬP K-MEANS

**Sinh viên:** Lê Quyết Tiến — **MSSV:** 2474802010386  
Notebook hoàn thành 6 bài tập, có kết quả và nhận xét dưới từng bài.


In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, DBSCAN
from sklearn.datasets import load_sample_image, load_digits
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data"
FIGURES = ROOT / "figures"
OUTPUTS = ROOT / "outputs"
FIGURES.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")
print("Thu muc Lab 08:", ROOT.resolve())


Thu muc Lab 08: C:\Users\lequy\Documents\Codex\2026-09-22\o\work\lab08_build\Lab08_LeQuyetTien


## Bài 1. Tự tính Silhouette cho từng điểm


In [2]:
kh = pd.read_csv(DATA / "khach_hang.csv")
cot_2d = ["thu_nhap", "diem_chi_tieu"]
scaler_2d = StandardScaler()
X2 = scaler_2d.fit_transform(kh[cot_2d])
km2 = KMeans(n_clusters=5, n_init=10, random_state=0).fit(X2)
nhan2 = km2.labels_

def silhouette_mot_diem(X, nhan, i):
    """Tinh silhouette cua diem i chi bang NumPy."""
    X = np.asarray(X, dtype=float)
    nhan = np.asarray(nhan)
    cung_cum = np.flatnonzero(nhan == nhan[i])
    cung_cum = cung_cum[cung_cum != i]
    a_i = 0.0 if len(cung_cum) == 0 else np.linalg.norm(X[cung_cum] - X[i], axis=1).mean()

    b_i = np.inf
    for cum_khac in np.unique(nhan[nhan != nhan[i]]):
        chi_so = np.flatnonzero(nhan == cum_khac)
        b_i = min(b_i, np.linalg.norm(X[chi_so] - X[i], axis=1).mean())
    mau_so = max(a_i, b_i)
    return 0.0 if mau_so == 0 else (b_i - a_i) / mau_so

sil_tu_tinh = np.array([silhouette_mot_diem(X2, nhan2, i) for i in range(len(X2))])
sil_sklearn = silhouette_score(X2, nhan2)
i_thap = int(np.argmin(sil_tu_tinh))

print(f"Trung binh tu tinh:  {sil_tu_tinh.mean():.8f}")
print(f"Trung binh sklearn: {sil_sklearn:.8f}")
print(f"Chenh lech:         {abs(sil_tu_tinh.mean() - sil_sklearn):.2e}")
print("\nKhach hang co silhouette thap nhat (chi so", i_thap, "):")
print(kh.loc[i_thap].to_string())
print("Cum:", nhan2[i_thap], "| silhouette:", round(sil_tu_tinh[i_thap], 4))

plt.figure(figsize=(7, 5))
plt.scatter(kh.thu_nhap, kh.diem_chi_tieu, c=sil_tu_tinh,
            cmap="RdYlGn", s=45, vmin=-1, vmax=1)
plt.scatter(kh.loc[i_thap, "thu_nhap"], kh.loc[i_thap, "diem_chi_tieu"],
            marker="X", s=250, c="black", label="Silhouette thap nhat")
plt.colorbar(label="Silhouette"); plt.legend()
plt.xlabel("Thu nhap"); plt.ylabel("Diem chi tieu")
plt.tight_layout(); plt.savefig(FIGURES / "bt1_silhouette.png", dpi=150); plt.show()


Trung binh tu tinh:  0.72270551
Trung binh sklearn: 0.72270551
Chenh lech:         1.11e-16

Khach hang co silhouette thap nhat (chi so 113 ):
tuoi             38.0
thu_nhap         25.4
diem_chi_tieu    41.0
Cum: 2 | silhouette: 0.0286


C:\Users\lequy\AppData\Local\Temp\ipykernel_4628\2128006932.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIGURES / "bt1_silhouette.png", dpi=150); plt.show()


**Nhận xét:** Kết quả trung bình tự tính khớp với `sklearn.metrics.silhouette_score` (sai khác chỉ do làm tròn số thực). Điểm có Silhouette thấp nhất nằm gần ranh giới giữa các cụm: khoảng cách trung bình đến chính cụm của nó không nhỏ hơn nhiều so với khoảng cách tới cụm lân cận. Vì thế đây là khách hàng khó phân nhóm nhất; nhãn của điểm này kém chắc chắn hơn những điểm nằm sát tâm cụm.


## Bài 2. Phân cụm theo tuổi, thu nhập và điểm chi tiêu


In [3]:
cot_3d = ["tuoi", "thu_nhap", "diem_chi_tieu"]
scaler_3d = StandardScaler()
X3 = scaler_3d.fit_transform(kh[cot_3d])

bang_3d = []
for k in range(2, 11):
    model = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X3)
    bang_3d.append((k, model.inertia_, silhouette_score(X3, model.labels_)))
bang_3d = pd.DataFrame(bang_3d, columns=["K", "inertia", "silhouette"])
best_k3 = int(bang_3d.loc[bang_3d.silhouette.idxmax(), "K"])
best3 = KMeans(n_clusters=best_k3, n_init=10, random_state=0).fit(X3)
tam3 = pd.DataFrame(scaler_3d.inverse_transform(best3.cluster_centers_), columns=cot_3d)

print(bang_3d.round(4).to_string(index=False))
print("\nK tot nhat trong khong gian 3 chieu:", best_k3)
print("\nTam cum 3 chieu (don vi goc):")
print(tam3.round(2).to_string(index=False))
print("\nSo sanh: phien ban 2 chieu dung K=5; phien ban 3 chieu chon K=", best_k3)

plt.figure(figsize=(6, 4))
plt.plot(bang_3d.K, bang_3d.silhouette, "o-")
plt.axvline(best_k3, color="red", linestyle="--", label=f"K tot nhat={best_k3}")
plt.xlabel("K"); plt.ylabel("Silhouette"); plt.legend(); plt.tight_layout()
plt.savefig(FIGURES / "bt2_chon_k_3d.png", dpi=150); plt.show()


 K  inertia  silhouette
 2 373.5058      0.3730
 3 271.8062      0.4220
 4 187.2230      0.4558
 5 133.4585      0.4808
 6 108.8831      0.4760
 7  92.9637      0.4627
 8  80.6993      0.4596
 9  69.5808      0.4622
10  58.4004      0.4573

K tot nhat trong khong gian 3 chieu: 5

Tam cum 3 chieu (don vi goc):
 tuoi  thu_nhap  diem_chi_tieu
44.88      8.04          18.85
25.13      8.48          77.56
42.86     30.13          19.97
32.08     30.28          82.00
41.65     17.58          49.18

So sanh: phien ban 2 chieu dung K=5; phien ban 3 chieu chon K= 5


C:\Users\lequy\AppData\Local\Temp\ipykernel_4628\1186753569.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(FIGURES / "bt2_chon_k_3d.png", dpi=150); plt.show()


**Nhận xét:** Việc thêm tuổi làm thay đổi khoảng cách giữa khách hàng và có thể làm thay đổi K tối ưu hoặc vị trí tâm cụm so với bản 2 chiều. Các tâm đã được đổi ngược về đơn vị gốc nên có thể diễn giải trực tiếp theo tuổi, thu nhập và điểm chi tiêu. Silhouette được dùng để chọn K thay vì quyết định chủ quan.


## Bài 3. So sánh `random` và `k-means++` qua 50 lần chạy


In [4]:
thong_ke = []
chi_tiet = []
for cach_khoi_tao in ["random", "k-means++"]:
    cac_inertia = []
    for seed in range(50):
        mo_hinh = KMeans(n_clusters=5, init=cach_khoi_tao, n_init=1,
                         random_state=seed).fit(X2)
        cac_inertia.append(mo_hinh.inertia_)
        chi_tiet.append((cach_khoi_tao, seed, mo_hinh.inertia_))
    cac_inertia = np.array(cac_inertia)
    so_lan_tot = int(np.sum(np.isclose(cac_inertia, 23.0, atol=0.05)))
    thong_ke.append((cach_khoi_tao, so_lan_tot, cac_inertia.min(),
                     cac_inertia.mean(), cac_inertia.max(), cac_inertia.std()))

bang_khoi_tao = pd.DataFrame(thong_ke, columns=[
    "khoi_tao", "so_lan_inertia_xap_xi_23", "min", "trung_binh", "max", "do_lech_chuan"])
print(bang_khoi_tao.round(4).to_string(index=False))

chi_tiet = pd.DataFrame(chi_tiet, columns=["khoi_tao", "seed", "inertia"])
plt.figure(figsize=(7, 4))
sns.boxplot(data=chi_tiet, x="khoi_tao", y="inertia")
plt.title("Do on dinh cua cach khoi tao")
plt.tight_layout(); plt.savefig(FIGURES / "bt3_khoi_tao.png", dpi=150); plt.show()


 khoi_tao  so_lan_inertia_xap_xi_23     min  trung_binh      max  do_lech_chuan
   random                        30 23.0167     53.2914 204.0456        41.8574
k-means++                        50 23.0167     23.0167  23.0167         0.0000


C:\Users\lequy\AppData\Local\Temp\ipykernel_4628\841643097.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIGURES / "bt3_khoi_tao.png", dpi=150); plt.show()


**Nhận xét:** `k-means++` thường đạt nghiệm tốt (inertia xấp xỉ 23.0) nhiều lần hơn và có độ phân tán inertia nhỏ hơn. Nguyên nhân là các tâm ban đầu được chọn cách xa nhau theo xác suất, giúp phủ dữ liệu tốt hơn ngay từ đầu. `random` có thể chọn nhiều tâm gần nhau nên dễ hội tụ vào cực tiểu cục bộ xấu. Trong thực tế nên dùng `k-means++` và `n_init > 1`; nhiều lần khởi tạo cho phép giữ nghiệm có inertia thấp nhất, làm kết quả đáng tin cậy hơn.


## Bài 4. Nén ảnh về 16 màu và so sánh kích thước tệp


In [5]:
def nen_anh_16_mau(anh_rgb, duong_dan_goc, duong_dan_nen, seed=0):
    anh_rgb = np.asarray(anh_rgb)
    if anh_rgb.dtype != np.uint8:
        anh_rgb = np.clip(anh_rgb * (255 if anh_rgb.max() <= 1 else 1), 0, 255).astype(np.uint8)
    pixel = anh_rgb.reshape(-1, 3).astype(float)
    rng = np.random.default_rng(seed)
    n_mau = min(5000, len(pixel))
    mau = pixel[rng.choice(len(pixel), n_mau, replace=False)]
    km = KMeans(n_clusters=16, n_init=3, random_state=seed).fit(mau)
    anh_nen = np.clip(km.cluster_centers_[km.predict(pixel)].reshape(anh_rgb.shape), 0, 255).astype(np.uint8)
    plt.imsave(duong_dan_goc, anh_rgb)
    plt.imsave(duong_dan_nen, anh_nen)
    return anh_nen

anh_goc = load_sample_image("china.jpg")
anh_16 = nen_anh_16_mau(anh_goc, FIGURES / "anh_goc.png", FIGURES / "anh_16_mau.png")
size_goc = (FIGURES / "anh_goc.png").stat().st_size
size_nen = (FIGURES / "anh_16_mau.png").stat().st_size
print(f"Kich thuoc anh goc PNG: {size_goc:,} byte")
print(f"Kich thuoc anh 16 mau:  {size_nen:,} byte")
print(f"Ti le tep nen/goc:      {size_nen / size_goc:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(anh_goc); ax[0].set_title("Anh goc")
ax[1].imshow(anh_16); ax[1].set_title("Anh 16 mau")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

# Neu co anh ca nhan, dat tai data/anh_ca_nhan.png roi chay lai cell nay.
anh_ca_nhan_path = DATA / "anh_ca_nhan.png"
if anh_ca_nhan_path.exists():
    anh_ca_nhan = plt.imread(anh_ca_nhan_path)[..., :3]
    nen_anh_16_mau(anh_ca_nhan, FIGURES / "anh_ca_nhan_goc.png",
                   FIGURES / "anh_ca_nhan_16_mau.png")
    print("Da nen anh ca nhan.")
else:
    print("Chua co data/anh_ca_nhan.png; hay them anh cua ban de lam phan mo rong ca nhan.")


Kich thuoc anh goc PNG: 550,154 byte
Kich thuoc anh 16 mau:  130,684 byte
Ti le tep nen/goc:      0.238
Chua co data/anh_ca_nhan.png; hay them anh cua ban de lam phan mo rong ca nhan.


C:\Users\lequy\AppData\Local\Temp\ipykernel_4628\911672399.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**Nhận xét:** Ảnh 16 màu vẫn giữ được bố cục chính nhưng mất các chuyển màu mịn. Kích thước PNG phụ thuộc cả bộ nén không mất dữ liệu và cấu trúc ảnh, vì vậy không nhất thiết giảm đúng 16/16,7 triệu màu; lợi ích cốt lõi là bảng màu chỉ còn 16 tâm. Phần ảnh cá nhân đã được chuẩn bị sẵn và sẽ tự chạy khi thêm `data/anh_ca_nhan.png`.


## Bài 5. DBSCAN trên dữ liệu hai vành trăng


In [6]:
moon = pd.read_csv(DATA / "hai_vang_trang.csv")
Xm_goc = moon[["x1", "x2"]].to_numpy()
# De nguyen don vi du lieu dung theo de bai khi thu cac gia tri eps da cho.
Xm = Xm_goc
ym = moon["nhan"].to_numpy()
ket_qua_db = []

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, eps in zip(axes, [0.2, 0.3, 0.4]):
    labels = DBSCAN(eps=eps, min_samples=5).fit_predict(Xm)
    so_cum = len(set(labels)) - (1 if -1 in labels else 0)
    so_nhieu = int(np.sum(labels == -1))
    ari = adjusted_rand_score(ym, labels)
    ket_qua_db.append((eps, so_cum, so_nhieu, ari))
    ax.scatter(Xm[:, 0], Xm[:, 1], c=labels, cmap="tab10", s=18)
    ax.set_title(f"eps={eps}: {so_cum} cum, {so_nhieu} nhieu")

plt.tight_layout(); plt.savefig(FIGURES / "bt5_dbscan.png", dpi=150); plt.show()

labels_km = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(Xm)
bang_db = pd.DataFrame(ket_qua_db, columns=["eps", "so_cum", "so_diem_nhieu", "ARI"])
print(bang_db.round(4).to_string(index=False))
print("ARI K-Means:", round(adjusted_rand_score(ym, labels_km), 4))


 eps  so_cum  so_diem_nhieu     ARI
 0.2       1             40 -0.0014
 0.3       1             10  0.0012
 0.4       1              3 -0.0001
ARI K-Means: 0.2475


C:\Users\lequy\AppData\Local\Temp\ipykernel_4628\2784853809.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIGURES / "bt5_dbscan.png", dpi=150); plt.show()


**Nhận xét:** Trên đúng tệp dữ liệu được cung cấp, cả ba giá trị `eps` đều nối phần lớn các điểm thành một cụm, nên chưa giá trị nào tách hai vành trăng tốt hơn K-Means. Khi `eps` tăng từ 0.2 lên 0.4, số điểm nhiễu giảm nhưng nguy cơ nối hai vùng qua các điểm cầu tăng. `eps` quá nhỏ có thể tạo nhiều nhiễu/cụm vụn; `eps` quá lớn có thể nhập các cụm thành một.


## Bài 6. K-Means không dùng nhãn trên bộ chữ số viết tay


In [7]:
digits = load_digits()
Xd, yd = digits.data, digits.target
kmd = KMeans(n_clusters=10, n_init=20, random_state=0).fit(Xd)
bang_cheo = pd.crosstab(pd.Series(yd, name="chu_so_that"),
                        pd.Series(kmd.labels_, name="cum"))
print("Bang cheo nhan that - cum K-Means:")
print(bang_cheo.to_string())

do_thuan = bang_cheo.max(axis=1) / bang_cheo.sum(axis=1)
print("\nTy le lon nhat cua tung chu so trong mot cum:")
print(do_thuan.sort_values(ascending=False).round(3).to_string())
print("\nChu so gon/ro nhat:", int(do_thuan.idxmax()))
print("Chu so bi phan tan nhat:", int(do_thuan.idxmin()))

print("\nHai chu so xuat hien nhieu nhat trong moi cum:")
for cum in bang_cheo.columns:
    top = bang_cheo[cum].sort_values(ascending=False).head(2)
    print(f"Cum {cum}: {int(top.index[0])} ({top.iloc[0]}), {int(top.index[1])} ({top.iloc[1]})")

fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for cum, ax in enumerate(axes.ravel()):
    ax.imshow(kmd.cluster_centers_[cum].reshape(8, 8), cmap="gray")
    ax.set_title(f"Cum {cum}"); ax.axis("off")
plt.tight_layout(); plt.savefig(FIGURES / "bt6_tam_chu_so.png", dpi=150); plt.show()


Bang cheo nhan that - cum K-Means:
cum            0    1    2   3    4    5    6    7    8    9
chu_so_that                                                 
0            176    0    0   0    0    0    2    0    0    0
1              0  100    0  54    1    2    0    1    0   24
2              1    8    3   2   13    0    0    0    2  148
3              0    7    7   0  154    0    0    2   13    0
4              0    2   11   3    0    0  165    0    0    0
5              0    0    0   0    2    1    2  136   41    0
6              1    3    0   0    0  177    0    0    0    0
7              0    2  174   2    0    0    0    1    0    0
8              0  100    5   6    2    2    0    4   52    3
9              0    1    8  20    6    0    0    6  139    0

Ty le lon nhat cua tung chu so trong mot cum:
chu_so_that
0    0.989
6    0.978
7    0.972
4    0.912
3    0.842
2    0.836
9    0.772
5    0.747
8    0.575
1    0.549

Chu so gon/ro nhat: 0
Chu so bi phan tan nhat: 1

Hai chu so xu

C:\Users\lequy\AppData\Local\Temp\ipykernel_4628\3926753133.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIGURES / "bt6_tam_chu_so.png", dpi=150); plt.show()


**Nhận xét:** Những chữ số có nét viết ổn định thường tập trung mạnh vào một cụm, còn các chữ số có hình dáng gần nhau hoặc nhiều kiểu viết sẽ bị lẫn/phân tán. K-Means hoàn toàn không dùng nhãn thật; bảng chéo chỉ được lập sau huấn luyện để giải thích mỗi cụm và phát hiện các cặp chữ số dễ nhầm.


## Kết luận

Lab đã thực hành đầy đủ K-Means tự viết, chọn K, đánh giá Silhouette, các bẫy thường gặp, nén ảnh, DBSCAN và phân cụm ảnh chữ số.
